<a href="https://colab.research.google.com/github/Sambarlasagna/Deep_reinf_learning/blob/main/ppo_lunaragent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install setuptools --upgrade


In [2]:
!apt install python-opengl -y
!apt install ffmpeg -y
!apt install xvfb -y
!apt install swig cmake -y
!pip install pyglet==1.5
!pip install pyvirtualdisplay

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
E: Unable to locate package python-opengl
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 41 not upgraded.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
xvfb is already the newest version (2:21.1.4-2ubuntu1.7~22.04.16).
0 upgraded, 0 newly installed, 0 to remove and 41 not upgraded.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
swig is already the newest version (4.0.2-1ubuntu1).
cmake is already the newest version (3.22.1-1ubuntu1.22.04.2).
0 upgraded, 0 newly installed, 0 to remove and 41 not upgraded.


In [3]:
from pyvirtualdisplay import Display

virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()

In [4]:
print("Installing gym and dependencies...")
!pip install -q gym==0.22
!pip install -q imageio-ffmpeg
!pip install -q huggingface_hub

# For box2d, we'll try a workaround since it has build issues
print("\nAttempting to install Box2D...")
try:
    !pip install box2d-py
    print("✓ Box2D installed successfully")
except:
    print("⚠ Box2D installation failed - continuing without it")
    print("  (Box2D is only needed for certain environments like LunarLander)")

print("\n✓ Installation complete!")

Installing gym and dependencies...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 631.1/631.1 kB 15.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done

Attempting to install Box2D...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.5/374.5 kB 10.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for box2d-py: filename=box2d_py-2.3.8-cp312-cp312-linux_x86_64.whl size=2382048 sha256=a8a2090c95d55e35e03896eb5c6f79c0afd5b00f01d6b121c0d22b98ef89e109
  Stored in directory: /root/.cache/pip/wheels/d6/3c/ab/b6fd75459cadc56f4a4125d4cb387a708a59ca8589e4cc6b7d
Successfully built box2d-py
✓ Box2D installed successfully

✓ Installation complete!


In [5]:
import argparse
import os
import random
import time

# Import gym first to check if it works
try:
    import gym
    print(f"✓ Gym {gym.__version__} imported successfully")
except Exception as e:
    print(f"⚠ Error importing gym: {e}")
    print("Trying to fix distutils issue...")

    # Try the distutils fix
    import sys
    try:
        from setuptools import _distutils
        sys.modules['distutils'] = _distutils
        sys.modules['distutils.util'] = _distutils.util
        import gym
        print("✓ Gym imported after distutils fix")
    except:
        print("❌ Could not import gym. Please restart runtime and try again.")
        raise

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions.categorical import Categorical
from torch.utils.tensorboard import SummaryWriter

from huggingface_hub import HfApi, upload_folder
from huggingface_hub.repocard import metadata_eval_result, metadata_save

from pathlib import Path
import datetime
import tempfile
import json
import shutil
import imageio

from wasabi import Printer
msg = Printer()

print("✓ All imports successful!")

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


✓ Gym 0.22.0 imported successfully
✓ All imports successful!


In [6]:
def package_to_hub(repo_id, model, hyperparameters, eval_env, video_fps=30,
                   commit_message="Push agent to the Hub", token=None, logs=None):
    """Evaluate, Generate a video and Upload a model to Hugging Face Hub."""
    msg.info(
        "This function will save, evaluate, generate a video of your agent, "
        "create a model card and push everything to the hub. "
        "It might take up to 1min. \n "
        "This is a work in progress: if you encounter a bug, please open an issue."
    )

    repo_url = HfApi().create_repo(
        repo_id=repo_id,
        token=token,
        private=False,
        exist_ok=True,
    )

    with tempfile.TemporaryDirectory() as tmpdirname:
        tmpdirname = Path(tmpdirname)

        torch.save(model.state_dict(), tmpdirname / "model.pt")

        mean_reward, std_reward = _evaluate_agent(eval_env, 10, model)

        eval_datetime = datetime.datetime.now()
        eval_form_datetime = eval_datetime.isoformat()

        evaluate_data = {
            "env_id": hyperparameters.env_id,
            "mean_reward": mean_reward,
            "std_reward": std_reward,
            "n_evaluation_episodes": 10,
            "eval_datetime": eval_form_datetime,
        }

        with open(tmpdirname / "results.json", "w") as outfile:
            json.dump(evaluate_data, outfile)

        video_path = tmpdirname / "replay.mp4"
        record_video(eval_env, model, video_path, video_fps)

        generated_model_card, metadata = _generate_model_card(
            "PPO", hyperparameters.env_id, mean_reward, std_reward, hyperparameters
        )
        _save_model_card(tmpdirname, generated_model_card, metadata)

        if logs:
            _add_logdir(tmpdirname, Path(logs))

        msg.info(f"Pushing repo {repo_id} to the Hugging Face Hub")

        repo_url = upload_folder(
            repo_id=repo_id,
            folder_path=tmpdirname,
            path_in_repo="",
            commit_message=commit_message,
            token=token,
        )

        msg.info(f"Your model is pushed to the Hub. You can view your model here: {repo_url}")
    return repo_url


def _evaluate_agent(env, n_eval_episodes, policy):
    """Evaluate the agent for n_eval_episodes episodes."""
    episode_rewards = []
    for episode in range(n_eval_episodes):
        state = env.reset()
        done = False
        total_rewards_ep = 0

        while not done:
            state = torch.Tensor(state).to(device)
            action, _, _, _ = policy.get_action_and_value(state)
            new_state, reward, done, info = env.step(action.cpu().numpy())
            total_rewards_ep += reward
            if done:
                break
            state = new_state
        episode_rewards.append(total_rewards_ep)

    mean_reward = np.mean(episode_rewards)
    std_reward = np.std(episode_rewards)
    return mean_reward, std_reward


def record_video(env, policy, out_directory, fps=30):
    """Record a video of the agent."""
    images = []
    done = False
    state = env.reset()
    img = env.render(mode='rgb_array')
    images.append(img)

    while not done:
        state = torch.Tensor(state).to(device)
        action, _, _, _ = policy.get_action_and_value(state)
        state, reward, done, info = env.step(action.cpu().numpy())
        img = env.render(mode='rgb_array')
        images.append(img)

    imageio.mimsave(out_directory, [np.array(img) for i, img in enumerate(images)], fps=fps)


def _generate_model_card(model_name, env_id, mean_reward, std_reward, hyperparameters):
    """Generate the model card for the Hub."""
    metadata = generate_metadata(model_name, env_id, mean_reward, std_reward)

    converted_dict = vars(hyperparameters)
    converted_str = str(converted_dict)
    converted_str = converted_str.split(", ")
    converted_str = '\n'.join(converted_str)

    model_card = f"""
# PPO Agent Playing {env_id}

This is a trained model of a PPO agent playing {env_id}.

# Hyperparameters
```python
{converted_str}
```
"""
    return model_card, metadata


def generate_metadata(model_name, env_id, mean_reward, std_reward):
    """Define the tags for the model card."""
    metadata = {}
    metadata["tags"] = [
        env_id,
        "ppo",
        "deep-reinforcement-learning",
        "reinforcement-learning",
        "custom-implementation",
        "deep-rl-course"
    ]

    eval = metadata_eval_result(
        model_pretty_name=model_name,
        task_pretty_name="reinforcement-learning",
        task_id="reinforcement-learning",
        metrics_pretty_name="mean_reward",
        metrics_id="mean_reward",
        metrics_value=f"{mean_reward:.2f} +/- {std_reward:.2f}",
        dataset_pretty_name=env_id,
        dataset_id=env_id,
    )

    metadata = {**metadata, **eval}
    return metadata


def _save_model_card(local_path, generated_model_card, metadata):
    """Save a model card for the repository."""
    readme_path = local_path / "README.md"
    readme = ""
    if readme_path.exists():
        with readme_path.open("r", encoding="utf8") as f:
            readme = f.read()
    else:
        readme = generated_model_card

    with readme_path.open("w", encoding="utf-8") as f:
        f.write(readme)

    metadata_save(readme_path, metadata)


def _add_logdir(local_path, logdir):
    """Add a logdir to the repository."""
    if logdir.exists() and logdir.is_dir():
        repo_logdir = local_path / "logs"
        if repo_logdir.exists():
            shutil.rmtree(repo_logdir)
        shutil.copytree(logdir, repo_logdir)

print("HuggingFace functions defined!")

HuggingFace functions defined!


In [7]:
# Define strtobool since it's removed in Python 3.12
def strtobool(val):
    """Convert a string representation of truth to true (1) or false (0)."""
    val = val.lower()
    if val in ('y', 'yes', 't', 'true', 'on', '1'):
        return 1
    elif val in ('n', 'no', 'f', 'false', 'off', '0'):
        return 0
    else:
        raise ValueError(f"invalid truth value {val}")

def make_env(env_id, seed, idx, capture_video, run_name):
    """Create a single environment."""
    def thunk():
        env = gym.make(env_id)
        env = gym.wrappers.RecordEpisodeStatistics(env)
        if capture_video:
            if idx == 0:
                env = gym.wrappers.RecordVideo(env, f"videos/{run_name}")
        env.seed(seed)
        env.action_space.seed(seed)
        env.observation_space.seed(seed)
        return env
    return thunk


def layer_init(layer, std=np.sqrt(2), bias_const=0.0):
    """Initialize layer with orthogonal weights."""
    torch.nn.init.orthogonal_(layer.weight, std)
    torch.nn.init.constant_(layer.bias, bias_const)
    return layer


class Agent(nn.Module):
    """PPO Agent with actor-critic architecture."""
    def __init__(self, envs):
        super().__init__()
        self.critic = nn.Sequential(
            layer_init(nn.Linear(np.array(envs.single_observation_space.shape).prod(), 64)),
            nn.Tanh(),
            layer_init(nn.Linear(64, 64)),
            nn.Tanh(),
            layer_init(nn.Linear(64, 1), std=1.0),
        )
        self.actor = nn.Sequential(
            layer_init(nn.Linear(np.array(envs.single_observation_space.shape).prod(), 64)),
            nn.Tanh(),
            layer_init(nn.Linear(64, 64)),
            nn.Tanh(),
            layer_init(nn.Linear(64, envs.single_action_space.n), std=0.01),
        )

    def get_value(self, x):
        return self.critic(x)

    def get_action_and_value(self, x, action=None):
        logits = self.actor(x)
        probs = Categorical(logits=logits)
        if action is None:
            action = probs.sample()
        return action, probs.log_prob(action), probs.entropy(), self.critic(x)

print("Agent class defined!")

# ============================================================================
# CELL 8: Define the training function
# ============================================================================

def train_ppo(env_id="LunarLander-v2", repo_id="YourUsername/ppo-LunarLander-v2",
              total_timesteps=500000, seed=1):
    """Train PPO agent."""

    # Create args object
    class Args:
        pass

    args = Args()
    args.exp_name = "ppo-lunarlander"
    args.seed = seed
    args.torch_deterministic = True
    args.cuda = True
    args.track = False
    args.wandb_project_name = "cleanRL"
    args.wandb_entity = None
    args.capture_video = False
    args.env_id = env_id
    args.total_timesteps = total_timesteps
    args.learning_rate = 2.5e-4
    args.num_envs = 4
    args.num_steps = 128
    args.anneal_lr = True
    args.gae = True
    args.gamma = 0.99
    args.gae_lambda = 0.95
    args.num_minibatches = 4
    args.update_epochs = 4
    args.norm_adv = True
    args.clip_coef = 0.2
    args.clip_vloss = True
    args.ent_coef = 0.01
    args.vf_coef = 0.5
    args.max_grad_norm = 0.5
    args.target_kl = None
    args.repo_id = repo_id
    args.batch_size = int(args.num_envs * args.num_steps)
    args.minibatch_size = int(args.batch_size // args.num_minibatches)

    run_name = f"{args.env_id}__{args.exp_name}__{args.seed}__{int(time.time())}"

    writer = SummaryWriter(f"runs/{run_name}")
    writer.add_text(
        "hyperparameters",
        "|param|value|\n|-|-|\n%s" % ("\n".join([f"|{key}|{value}|" for key, value in vars(args).items()])),
    )

    # Seeding
    random.seed(args.seed)
    np.random.seed(args.seed)
    torch.manual_seed(args.seed)
    torch.backends.cudnn.deterministic = args.torch_deterministic

    global device
    device = torch.device("cuda" if torch.cuda.is_available() and args.cuda else "cpu")
    print(f"Using device: {device}")

    # Environment setup
    envs = gym.vector.SyncVectorEnv(
        [make_env(args.env_id, args.seed + i, i, args.capture_video, run_name) for i in range(args.num_envs)]
    )

    agent = Agent(envs).to(device)
    optimizer = optim.Adam(agent.parameters(), lr=args.learning_rate, eps=1e-5)

    # Storage setup
    obs = torch.zeros((args.num_steps, args.num_envs) + envs.single_observation_space.shape).to(device)
    actions = torch.zeros((args.num_steps, args.num_envs) + envs.single_action_space.shape).to(device)
    logprobs = torch.zeros((args.num_steps, args.num_envs)).to(device)
    rewards = torch.zeros((args.num_steps, args.num_envs)).to(device)
    dones = torch.zeros((args.num_steps, args.num_envs)).to(device)
    values = torch.zeros((args.num_steps, args.num_envs)).to(device)

    # Start training
    global_step = 0
    start_time = time.time()
    next_obs = torch.Tensor(envs.reset()).to(device)
    next_done = torch.zeros(args.num_envs).to(device)
    num_updates = args.total_timesteps // args.batch_size

    print(f"Starting training for {num_updates} updates...")

    for update in range(1, num_updates + 1):
        # Annealing learning rate
        if args.anneal_lr:
            frac = 1.0 - (update - 1.0) / num_updates
            lrnow = frac * args.learning_rate
            optimizer.param_groups[0]["lr"] = lrnow

        for step in range(0, args.num_steps):
            global_step += 1 * args.num_envs
            obs[step] = next_obs
            dones[step] = next_done

            with torch.no_grad():
                action, logprob, _, value = agent.get_action_and_value(next_obs)
                values[step] = value.flatten()
            actions[step] = action
            logprobs[step] = logprob

            next_obs, reward, done, info = envs.step(action.cpu().numpy())
            rewards[step] = torch.tensor(reward).to(device).view(-1)
            next_obs, next_done = torch.Tensor(next_obs).to(device), torch.Tensor(done).to(device)

            for item in info:
                if "episode" in item.keys():
                    print(f"global_step={global_step}, episodic_return={item['episode']['r']:.2f}")
                    writer.add_scalar("charts/episodic_return", item["episode"]["r"], global_step)
                    writer.add_scalar("charts/episodic_length", item["episode"]["l"], global_step)
                    break

        # Bootstrap value
        with torch.no_grad():
            next_value = agent.get_value(next_obs).reshape(1, -1)
            if args.gae:
                advantages = torch.zeros_like(rewards).to(device)
                lastgaelam = 0
                for t in reversed(range(args.num_steps)):
                    if t == args.num_steps - 1:
                        nextnonterminal = 1.0 - next_done
                        nextvalues = next_value
                    else:
                        nextnonterminal = 1.0 - dones[t + 1]
                        nextvalues = values[t + 1]
                    delta = rewards[t] + args.gamma * nextvalues * nextnonterminal - values[t]
                    advantages[t] = lastgaelam = delta + args.gamma * args.gae_lambda * nextnonterminal * lastgaelam
                returns = advantages + values
            else:
                returns = torch.zeros_like(rewards).to(device)
                for t in reversed(range(args.num_steps)):
                    if t == args.num_steps - 1:
                        nextnonterminal = 1.0 - next_done
                        next_return = next_value
                    else:
                        nextnonterminal = 1.0 - dones[t + 1]
                        next_return = returns[t + 1]
                    returns[t] = rewards[t] + args.gamma * nextnonterminal * next_return
                advantages = returns - values

        # Flatten the batch
        b_obs = obs.reshape((-1,) + envs.single_observation_space.shape)
        b_logprobs = logprobs.reshape(-1)
        b_actions = actions.reshape((-1,) + envs.single_action_space.shape)
        b_advantages = advantages.reshape(-1)
        b_returns = returns.reshape(-1)
        b_values = values.reshape(-1)

        # Optimizing the policy and value network
        b_inds = np.arange(args.batch_size)
        clipfracs = []
        for epoch in range(args.update_epochs):
            np.random.shuffle(b_inds)
            for start in range(0, args.batch_size, args.minibatch_size):
                end = start + args.minibatch_size
                mb_inds = b_inds[start:end]

                _, newlogprob, entropy, newvalue = agent.get_action_and_value(b_obs[mb_inds], b_actions.long()[mb_inds])
                logratio = newlogprob - b_logprobs[mb_inds]
                ratio = logratio.exp()

                with torch.no_grad():
                    old_approx_kl = (-logratio).mean()
                    approx_kl = ((ratio - 1) - logratio).mean()
                    clipfracs += [((ratio - 1.0).abs() > args.clip_coef).float().mean().item()]

                mb_advantages = b_advantages[mb_inds]
                if args.norm_adv:
                    mb_advantages = (mb_advantages - mb_advantages.mean()) / (mb_advantages.std() + 1e-8)

                # Policy loss
                pg_loss1 = -mb_advantages * ratio
                pg_loss2 = -mb_advantages * torch.clamp(ratio, 1 - args.clip_coef, 1 + args.clip_coef)
                pg_loss = torch.max(pg_loss1, pg_loss2).mean()

                # Value loss
                newvalue = newvalue.view(-1)
                if args.clip_vloss:
                    v_loss_unclipped = (newvalue - b_returns[mb_inds]) ** 2
                    v_clipped = b_values[mb_inds] + torch.clamp(
                        newvalue - b_values[mb_inds],
                        -args.clip_coef,
                        args.clip_coef,
                    )
                    v_loss_clipped = (v_clipped - b_returns[mb_inds]) ** 2
                    v_loss_max = torch.max(v_loss_unclipped, v_loss_clipped)
                    v_loss = 0.5 * v_loss_max.mean()
                else:
                    v_loss = 0.5 * ((newvalue - b_returns[mb_inds]) ** 2).mean()

                entropy_loss = entropy.mean()
                loss = pg_loss - args.ent_coef * entropy_loss + v_loss * args.vf_coef

                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(agent.parameters(), args.max_grad_norm)
                optimizer.step()

            if args.target_kl is not None:
                if approx_kl > args.target_kl:
                    break

        y_pred, y_true = b_values.cpu().numpy(), b_returns.cpu().numpy()
        var_y = np.var(y_true)
        explained_var = np.nan if var_y == 0 else 1 - np.var(y_true - y_pred) / var_y

        # Record rewards
        writer.add_scalar("charts/learning_rate", optimizer.param_groups[0]["lr"], global_step)
        writer.add_scalar("losses/value_loss", v_loss.item(), global_step)
        writer.add_scalar("losses/policy_loss", pg_loss.item(), global_step)
        writer.add_scalar("losses/entropy", entropy_loss.item(), global_step)
        writer.add_scalar("losses/old_approx_kl", old_approx_kl.item(), global_step)
        writer.add_scalar("losses/approx_kl", approx_kl.item(), global_step)
        writer.add_scalar("losses/clipfrac", np.mean(clipfracs), global_step)
        writer.add_scalar("losses/explained_variance", explained_var, global_step)

        if update % 10 == 0:
            sps = int(global_step / (time.time() - start_time))
            print(f"Update {update}/{num_updates}, SPS: {sps}")
            writer.add_scalar("charts/SPS", sps, global_step)

    envs.close()
    writer.close()

    print("\n🎉 Training complete! Now uploading to HuggingFace...")

    # Upload to HuggingFace
    eval_env = gym.make(args.env_id)
    package_to_hub(
        repo_id=args.repo_id,
        model=agent,
        hyperparameters=args,
        eval_env=eval_env,
        logs=f"runs/{run_name}",
    )

    return agent, args

print("Training function defined!")

Agent class defined!
Training function defined!


In [8]:
from huggingface_hub import notebook_login
notebook_login()


In [9]:
agent, args = train_ppo(
    env_id="LunarLander-v2",
    repo_id="Sambarlasagna/ppo-LunarLander-v2",  # 👈 CHANGE THIS!
    total_timesteps=500000,  # Reduce to 50000 for quick test
    seed=1
)

print("\n✅ All done! Check your HuggingFace profile for the model.")

Using device: cuda


/usr/local/lib/python3.12/dist-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


Starting training for 976 updates...
global_step=424, episodic_return=-170.82
global_step=472, episodic_return=-320.47
global_step=476, episodic_return=-81.00
global_step=484, episodic_return=-20.28
global_step=700, episodic_return=-95.84
global_step=784, episodic_return=-81.62
global_step=844, episodic_return=-123.97
global_step=948, episodic_return=-176.86
global_step=1000, episodic_return=-130.59
global_step=1136, episodic_return=-112.11
global_step=1188, episodic_return=-513.12
global_step=1236, episodic_return=-97.31
global_step=1352, episodic_return=-254.07
global_step=1404, episodic_return=-155.41
global_step=1624, episodic_return=-161.66
global_step=1712, episodic_return=-250.64
global_step=1752, episodic_return=-285.36
global_step=1816, episodic_return=-151.20
global_step=1888, episodic_return=-66.42
global_step=2004, episodic_return=-85.49
global_step=2080, episodic_return=-77.72
global_step=2240, episodic_return=-95.56
global_step=2276, episodic_return=-68.40
global_step=241

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


ℹ Pushing repo Sambarlasagna/ppo-LunarLander-v2 to the Hugging Face
Hub


...

New Data Upload               : |          |  0.00B /  0.00B            

  /tmp/tmpnam9l8as/model.pt   : 100%|##########| 43.4kB / 43.4kB            

  ...45487.c0cbfc830103.2576.0: 100%|##########|  631kB /  631kB            

  /tmp/tmpnam9l8as/replay.mp4 : 100%|##########|  210kB /  210kB            

ℹ Your model is pushed to the Hub. You can view your model here:
https://huggingface.co/Sambarlasagna/ppo-LunarLander-v2/tree/main/

✅ All done! Check your HuggingFace profile for the model.
